# Step 1b (Option B) — Institution Labeling Pilot

**Goal of this notebook:** rebuild institution labels the Option B way.
We do NOT re-label the whole corpus. Instead we label two small sets with one
identical method, so fairness metrics are clean:
1. a random baseline sample of the corpus (unbiased corpus elite share)
2. (later) the retrieved set from Step 5a

**This notebook is the PILOT.** It runs on only 300 random papers first, just to
measure how much affiliation coverage the method gets before we commit to the
full run.

**Method:** OpenAlex lookup by arXiv id (landing_page_url), with a DOI fallback
(`10.48550/arXiv.<id>`) for papers not found the first way. Elite = affiliation
matches the QS World University Rankings 2026 Top 50 list.

**Cell 1 does:** imports, config (paths, email), and loads the two input files
(corpus + elite list). It prints the real Kaggle paths so we can confirm them.

In [1]:
# Step 1b Option B - PILOT - Cell 1: setup and load inputs
# Runs on Kaggle. Reads two Kaggle datasets:
#   - corpus:      fairsearch-qbio-processed-Raj-Jici-YB  (qbio_papers.json)
#   - elite list:  fairsearch-qbio-elite-list             (qs_top50_elite_2026.json)

import json
import os
import glob
import random

# --- Config ---
MAILTO = "chen.yanbo@northeastern.edu"   # OpenAlex polite pool
PILOT_N = 300                            # pilot sample size
RANDOM_SEED = 42                         # fixed for reproducibility

# --- Expected Kaggle paths (may differ; auto-detect below if not found) ---
CORPUS_PATH_EXPECTED = "/kaggle/input/fairsearch-qbio-processed-Raj-Jici-YB/qbio_papers.json"
ELITE_PATH_EXPECTED  = "/kaggle/input/fairsearch-qbio-elite-list/qs_top50_elite_2026.json"

def find_file(expected_path, filename_hint):
    """Return expected_path if it exists, else search /kaggle/input for the file."""
    if os.path.exists(expected_path):
        return expected_path
    hits = glob.glob(f"/kaggle/input/**/{filename_hint}", recursive=True)
    if hits:
        print(f"[auto-detect] expected not found, using: {hits[0]}")
        return hits[0]
    # last resort: list everything so we can see what's actually mounted
    print(f"[ERROR] could not find {filename_hint}. Files under /kaggle/input:")
    for p in glob.glob("/kaggle/input/**/*", recursive=True):
        print("   ", p)
    raise FileNotFoundError(filename_hint)

CORPUS_PATH = find_file(CORPUS_PATH_EXPECTED, "qbio_papers.json")
ELITE_PATH  = find_file(ELITE_PATH_EXPECTED,  "qs_top50_elite_2026.json")

print("CORPUS_PATH =", CORPUS_PATH)
print("ELITE_PATH  =", ELITE_PATH)

# --- Load ---
with open(CORPUS_PATH) as f:
    corpus = json.load(f)
with open(ELITE_PATH) as f:
    elite_list = json.load(f)

print("corpus papers:", len(corpus))
print("elite institutions:", len(elite_list))
print("corpus sample record:", corpus[0])
print("elite sample record:", elite_list[0])

[auto-detect] expected not found, using: /kaggle/input/datasets/yanbochen928/fairsearch-qbio-processed-raj-jici-yb/qbio_papers.json
[auto-detect] expected not found, using: /kaggle/input/datasets/yanbochen928/qs-top50-elite-2026/qs_top50_elite_2026.json
CORPUS_PATH = /kaggle/input/datasets/yanbochen928/fairsearch-qbio-processed-raj-jici-yb/qbio_papers.json
ELITE_PATH  = /kaggle/input/datasets/yanbochen928/qs-top50-elite-2026/qs_top50_elite_2026.json
corpus papers: 55301
elite institutions: 50
corpus sample record: {'paper_id': '0704.0021', 'title': 'Molecular Synchronization Waves in Arrays of Allosterically Regulated Enzymes', 'abstract': 'Spatiotemporal pattern formation in a product-activated enzymic reaction at high enzyme concentrations is investigated. Stochastic simulations show that catalytic turnover cycles of individual enzymes can become coherent and that complex wave patterns of molecular synchronization can develop. The analysis based on the mean-field approximation indica

## Cell 2 — Draw the random pilot sample (300 papers)

We draw 300 papers at random from the corpus, with a fixed seed so the sample is
reproducible. We also classify each paper_id as "new format" (e.g. 1610.07213)
or "old format" (e.g. q-bio/0404009), because the DOI fallback may behave
differently for the two. This lets us read coverage separately for each format
later.

This cell does NOT call OpenAlex yet. It only selects the sample and shows the
new/old split.

In [2]:
# Step 1b Option B - PILOT - Cell 2: draw random 300-paper sample + id format split

# Reproducible random sample
random.seed(RANDOM_SEED)
sample = random.sample(corpus, PILOT_N)
print(f"drew {len(sample)} papers (seed={RANDOM_SEED})")

def id_format(paper_id):
    """Classify arXiv id format.
    'old' = has a category prefix with a slash, e.g. q-bio/0404009
    'new' = numeric YYMM.number, e.g. 1610.07213
    """
    return "old" if "/" in paper_id else "new"

# Count the new/old split in the sample
split = {"new": 0, "old": 0}
for p in sample:
    split[id_format(p["paper_id"])] += 1

print("id format split in sample:")
print("   new format (e.g. 1610.07213):", split["new"])
print("   old format (e.g. q-bio/0404009):", split["old"])

# Show a couple of example ids of each kind so we can eyeball them
examples_new = [p["paper_id"] for p in sample if id_format(p["paper_id"]) == "new"][:3]
examples_old = [p["paper_id"] for p in sample if id_format(p["paper_id"]) == "old"][:3]
print("   new examples:", examples_new)
print("   old examples:", examples_old)

drew 300 papers (seed=42)
id format split in sample:
   new format (e.g. 1610.07213): 282
   old format (e.g. q-bio/0404009): 18
   new examples: ['2409.04321', '1301.7745', '0812.3898']
   old examples: ['physics/0310009', 'physics/0110075', 'physics/0208040']


## Cell 3 — OpenAlex lookup function (with DOI fallback), tested on 3 papers

This defines how we ask OpenAlex for a paper's institution:

1. **Primary lookup** — query the OpenAlex `works` endpoint filtered by
   `locations.landing_page_url` = `arxiv.org/abs/<id>`.
2. **DOI fallback** — if the primary lookup finds nothing, try again by DOI
   `10.48550/arXiv.<id>`.

For a found work we read the first author's institution (display name + country).
We return one of three coverage states, matching Raj v1:
- `found`         : an affiliation was retrieved
- `no_affiliation`: the work exists but has no institution attached
- `not_found`     : OpenAlex returned nothing, even after the DOI fallback

This cell tests the function on just 3 papers so we can see the shape of the
result before running all 300. It is deliberately slow and polite (1 request at
a time).

In [3]:
# Step 1b Option B - PILOT - Cell 3: OpenAlex lookup function + 3-paper test
# FIX: OpenAlex stores arXiv landing_page_url as http:// (not https://)

import requests
import time

OPENALEX_WORKS = "https://api.openalex.org/works"

def _first_institution(work):
    """Extract (institution_display_name, country_code) from an OpenAlex work.
    Returns (None, None) if no institution is attached."""
    for authorship in work.get("authorships", []):
        for inst in authorship.get("institutions", []):
            name = inst.get("display_name")
            country = inst.get("country_code")
            if name:
                return name, country
    return None, None

def lookup_openalex(paper_id):
    """Look up one arXiv paper on OpenAlex.
    Returns dict: {coverage, institution, country, matched_by}."""
    base_params = {"mailto": MAILTO, "per-page": 1}

    # --- 1) Primary: by landing_page_url (OpenAlex stores http://, not https://) ---
    url_filter = f"locations.landing_page_url:http://arxiv.org/abs/{paper_id}"
    try:
        r = requests.get(OPENALEX_WORKS,
                         params={**base_params, "filter": url_filter},
                         timeout=30)
        r.raise_for_status()
        results = r.json().get("results", [])
    except Exception as e:
        results = []
        print(f"   [warn] primary lookup error for {paper_id}: {e}")

    matched_by = "landing_page_url"

    # --- 2) Fallback: by DOI 10.48550/arXiv.<id> ---
    if not results:
        doi = f"10.48550/arXiv.{paper_id}"
        doi_filter = f"doi:{doi}"
        try:
            r = requests.get(OPENALEX_WORKS,
                             params={**base_params, "filter": doi_filter},
                             timeout=30)
            r.raise_for_status()
            results = r.json().get("results", [])
            matched_by = "doi_fallback"
        except Exception as e:
            print(f"   [warn] doi fallback error for {paper_id}: {e}")

    # --- interpret result ---
    if not results:
        return {"coverage": "not_found", "institution": None,
                "country": None, "matched_by": None}

    name, country = _first_institution(results[0])
    if name is None:
        return {"coverage": "no_affiliation", "institution": None,
                "country": None, "matched_by": matched_by}
    return {"coverage": "found", "institution": name,
            "country": country, "matched_by": matched_by}

# --- Test on 3 papers (mix of new + old format) ---
test_ids = ["2409.04321", "0812.3898", "physics/0310009"]
for pid in test_ids:
    result = lookup_openalex(pid)
    print(pid, "->", result)
    time.sleep(1)   # be polite: 1 second between requests

2409.04321 -> {'coverage': 'no_affiliation', 'institution': None, 'country': None, 'matched_by': 'landing_page_url'}
0812.3898 -> {'coverage': 'found', 'institution': 'Foundation for Research and Technology Hellas', 'country': 'GR', 'matched_by': 'landing_page_url'}
physics/0310009 -> {'coverage': 'no_affiliation', 'institution': None, 'country': None, 'matched_by': 'landing_page_url'}


## Cell 3.1 — Diagnostic: inspect one paper's raw OpenAlex record

The 3-paper test showed primary (landing_page_url) lookups all missing, with DOI
fallback carrying everything, plus several no_affiliation. That pattern suggests
our primary query URL or our institution-parsing may be wrong, not that the data
is missing. Before running 300, we inspect one raw OpenAlex record to see the
actual landing_page_url format and where affiliation data really lives.

This is a throwaway diagnostic cell. It does not change anything.

In [4]:
# Step 1b Option B - PILOT - Cell 3.5: diagnostic on one paper's raw OpenAlex data

diag_id = "2409.04321"   # a recent paper, likely well-indexed

# Try to fetch the work by DOI (we know this one is findable via DOI)
doi = f"10.48550/arXiv.{diag_id}"
r = requests.get(OPENALEX_WORKS,
                 params={"mailto": MAILTO, "filter": f"doi:{doi}", "per-page": 1},
                 timeout=30)
r.raise_for_status()
results = r.json().get("results", [])
print("results found:", len(results))

if results:
    w = results[0]
    print("openalex id     :", w.get("id"))
    print("title           :", (w.get("title") or "")[:80])
    print("doi             :", w.get("doi"))

    # What landing page urls does OpenAlex actually store for this work?
    print("\nlocations landing_page_url values:")
    for loc in w.get("locations", []):
        print("   ", repr(loc.get("landing_page_url")))

    # How is affiliation stored? Inspect the first authorship in full.
    auths = w.get("authorships", [])
    print("\nnum authorships:", len(auths))
    if auths:
        a0 = auths[0]
        print("first author    :", a0.get("author", {}).get("display_name"))
        print("raw_affiliation_strings:", a0.get("raw_affiliation_strings"))
        print("institutions    :", a0.get("institutions"))

results found: 1
openalex id     : https://openalex.org/W4403587706
title           : The Effects of Unilateral Slope Loading on Lower Limb Plantar Flexor Muscle EMG 
doi             : https://doi.org/10.48550/arxiv.2409.04321

locations landing_page_url values:
    'http://arxiv.org/abs/2409.04321'
    'https://doi.org/10.48550/arxiv.2409.04321'

num authorships: 3
first author    : Xinyu Zhou
raw_affiliation_strings: []
institutions    : []


## Cell 4 — Run the lookup on all 300 sampled papers

Now we call the (fixed) lookup on every paper in the sample and record the
result for each. This is the slow step: one polite request per paper, so ~300
requests. Progress prints every 25 papers.

For each paper we store: paper_id, id_format (new/old), coverage, institution,
country, matched_by. Nothing is judged elite yet - that happens in Cell 5.

In [5]:
# Step 1b Option B - PILOT - Cell 4: run lookup on all 300 sampled papers

results_300 = []
t_start = time.time()

for i, p in enumerate(sample, start=1):
    pid = p["paper_id"]
    res = lookup_openalex(pid)
    res["paper_id"] = pid
    res["id_format"] = id_format(pid)
    results_300.append(res)

    time.sleep(0.2)  # polite pace; OpenAlex polite pool tolerates this

    if i % 25 == 0:
        elapsed = time.time() - t_start
        print(f"   {i}/{PILOT_N} done ({elapsed:.0f}s elapsed)")

print(f"\nfinished {len(results_300)} lookups in {time.time()-t_start:.0f}s")

# Quick coverage tally
from collections import Counter
cov = Counter(r["coverage"] for r in results_300)
print("coverage tally:", dict(cov))

   25/300 done (13s elapsed)
   50/300 done (27s elapsed)
   75/300 done (40s elapsed)
   100/300 done (53s elapsed)
   125/300 done (67s elapsed)
   150/300 done (79s elapsed)
   175/300 done (92s elapsed)
   200/300 done (105s elapsed)
   225/300 done (118s elapsed)
   250/300 done (131s elapsed)
   275/300 done (143s elapsed)
   300/300 done (156s elapsed)

finished 300 lookups in 156s
coverage tally: {'no_affiliation': 156, 'found': 138, 'not_found': 6}


## Cell 5 — Coverage by id format + elite share of the sample

Two things:
1. Break coverage down by id format (new vs old), to see if old-format ids are a
   separate weak spot.
2. Apply the QS Top-50 elite list to the `found` papers and compute the elite
   share of this random sample. This sample elite share is the UNBIASED BASELINE
   that Option B's SPD uses (SPD = retrieved_share - sample_share).

Matching rule for now: exact, case-insensitive match of the OpenAlex institution
display name against the QS Top-50 names. (We may add aliases later if needed.)

In [6]:
# Step 1b Option B - PILOT - Cell 5: coverage by id format + elite share

# --- 1) Coverage split by id format ---
from collections import defaultdict
by_fmt = defaultdict(Counter)
for r in results_300:
    by_fmt[r["id_format"]][r["coverage"]] += 1

print("coverage by id format:")
for fmt in ["new", "old"]:
    c = by_fmt[fmt]
    total = sum(c.values())
    found = c.get("found", 0)
    if total:
        print(f"   {fmt}: total={total}, found={found} ({found/total:.0%}), "
              f"no_affiliation={c.get('no_affiliation',0)}, "
              f"not_found={c.get('not_found',0)}")

# --- 2) Build a normalized elite name set ---
def norm(name):
    return name.strip().lower() if name else ""

elite_names = {norm(e["name"]) for e in elite_list}

# --- 3) Elite share among the 'found' papers ---
found_recs = [r for r in results_300 if r["coverage"] == "found"]
elite_count = 0
for r in found_recs:
    r["elite_label"] = 1 if norm(r["institution"]) in elite_names else 0
    elite_count += r["elite_label"]

n_found = len(found_recs)
print("\nelite share of the random sample (baseline):")
print(f"   found papers        : {n_found}")
print(f"   elite (QS Top-50)   : {elite_count}")
if n_found:
    print(f"   sample elite share  : {elite_count/n_found:.3f}")

# Show which elite institutions showed up, for a sanity check
elite_hits = Counter(r["institution"] for r in found_recs if r["elite_label"] == 1)
print("\nelite institutions matched in sample:")
for name, cnt in elite_hits.most_common():
    print(f"   {cnt}x  {name}")

coverage by id format:
   new: total=282, found=125 (44%), no_affiliation=151, not_found=6
   old: total=18, found=13 (72%), no_affiliation=5, not_found=0

elite share of the random sample (baseline):
   found papers        : 138
   elite (QS Top-50)   : 25
   sample elite share  : 0.181

elite institutions matched in sample:
   3x  University of Oxford
   3x  University of Pennsylvania
   2x  The University of Queensland
   2x  University of Cambridge
   2x  Princeton University
   2x  Yale University
   2x  Imperial College London
   2x  University of British Columbia
   1x  University of Chicago
   1x  Technical University of Munich
   1x  National University of Singapore
   1x  University of California, Los Angeles
   1x  Massachusetts Institute of Technology
   1x  ETH Zurich
   1x  Seoul National University
